# TimeSformer — From Scratch (V2)

**Divided Space-Time Attention** for video classification, implemented from scratch.

### V2 Improvements over V1:
- 5 classes (bench pressing, deadlifting, pull ups, push up, squat)
- Pre-LN Transformer with **dropout** (0.15) regularization
- **Full depth (12 layers)** — preserves all pretrained semantic features
- **Moderate augmentation** — RandomResizedCrop + ColorJitter + HorizontalFlip
- 2-stage fine-tuning (freeze → unfreeze)
- Label smoothing (0.1) + cosine LR scheduler + early stopping (patience 7)
- **Weight decay 0.05** for stronger L2 regularization

Pretrained backbone: ViT-Base/16 (ImageNet) — all 12 blocks

In [ ]:
!pip install torchvision timm einops -q

In [ ]:
# ── Imports ──

import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from einops import rearrange
from PIL import Image
import matplotlib.pyplot as plt
import timm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 1. Dataset — `VideoFrameDataset`

- Uniformly samples `num_frames` frames from each video
- **Moderate augmentation**: RandomResizedCrop + HorizontalFlip + ColorJitter
- ImageNet normalization for pretrained ViT compatibility

In [ ]:
class VideoFrameDataset(Dataset):
    def __init__(self, root_dir, num_frames=8, image_size=224,
                 split='train', val_ratio=0.2, seed=42):
        self.root_dir = root_dir
        self.num_frames = num_frames
        self.image_size = image_size

        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        # Collect all samples
        all_samples = []
        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for video in sorted(os.listdir(cls_path)):
                video_path = os.path.join(cls_path, video)
                if os.path.isdir(video_path):
                    all_samples.append((video_path, self.class_to_idx[cls]))

        # Train/Val split (deterministic)
        rng = random.Random(seed)
        indices = list(range(len(all_samples)))
        rng.shuffle(indices)
        split_idx = int(len(indices) * (1 - val_ratio))

        if split == 'train':
            self.samples = [all_samples[i] for i in indices[:split_idx]]
        else:
            self.samples = [all_samples[i] for i in indices[split_idx:]]

        # ImageNet normalization (essential for pretrained ViT)
        normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        if split == 'train':
            # ── MODERATE AUGMENTATION ──
            # RandomResizedCrop is the most effective single augmentation
            # for preventing overfitting on small datasets
            self.transform = transforms.Compose([
                transforms.RandomResizedCrop(
                    image_size, scale=(0.8, 1.0), ratio=(0.9, 1.1)
                ),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(
                    brightness=0.2, contrast=0.2, saturation=0.2
                ),
                transforms.ToTensor(),
                normalize,
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                normalize,
            ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frames = sorted(os.listdir(video_path))

        # Repeat frames if fewer than needed
        if len(frames) < self.num_frames:
            frames = frames * (self.num_frames // len(frames) + 1)

        # Uniform temporal sampling
        idxs = torch.linspace(0, len(frames) - 1, self.num_frames).long()
        selected = [frames[i] for i in idxs]

        imgs = []
        for f in selected:
            img = Image.open(os.path.join(video_path, f)).convert("RGB")
            imgs.append(self.transform(img))

        # Stack ALL frames — must be outside the loop!
        video = torch.stack(imgs)  # (T, C, H, W)
        return video, label

## 2. Architecture — TimeSformer (Pre-LN, depth=12)

**Pre-LN pattern:** `x = x + dropout(sub_block(norm(x)))`

Full 12-block depth preserves all pretrained semantic features.
Dropout (0.15) + weight decay handle regularization instead of reducing depth.

In [ ]:
class TimeSformerBlock(nn.Module):
    """Divided Space-Time Attention Block (Pre-LN + Dropout)."""

    def __init__(self, embed_dim, num_heads, dropout=0.15):
        super().__init__()

        # Pre-LN: norms come BEFORE each sub-block
        self.norm1 = nn.LayerNorm(embed_dim)  # before temporal attn
        self.norm2 = nn.LayerNorm(embed_dim)  # before spatial attn
        self.norm3 = nn.LayerNorm(embed_dim)  # before MLP

        self.temporal_attn = nn.MultiheadAttention(
            embed_dim, num_heads, batch_first=True, dropout=dropout
        )
        self.spatial_attn = nn.MultiheadAttention(
            embed_dim, num_heads, batch_first=True, dropout=dropout
        )

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

        # Dropout after each sub-block (before residual add)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x):
        b, t, n, d = x.shape

        # ── Temporal Attention (Pre-LN) ──
        xt = self.norm1(x)
        xt = rearrange(xt, "b t n d -> (b n) t d")
        xt = self.temporal_attn(xt, xt, xt)[0]
        xt = rearrange(xt, "(b n) t d -> b t n d", b=b)
        x = x + self.drop1(xt)

        # ── Spatial Attention (Pre-LN) ──
        xs = self.norm2(x)
        xs = rearrange(xs, "b t n d -> (b t) n d")
        xs = self.spatial_attn(xs, xs, xs)[0]
        xs = rearrange(xs, "(b t) n d -> b t n d", b=b)
        x = x + self.drop2(xs)

        # ── MLP (Pre-LN) ──
        x = x + self.mlp(self.norm3(x))

        return x

In [ ]:
class TimeSformer(nn.Module):
    """TimeSformer: Divided Space-Time Attention for Video Understanding."""

    def __init__(self, num_classes, num_frames=8, img_size=224,
                 patch_size=16, embed_dim=768, depth=12,
                 num_heads=12, dropout=0.15):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.num_frames = num_frames
        self.embed_dim = embed_dim

        # Patch embedding (Conv2d acts as linear projection)
        self.patch_embed = nn.Conv2d(
            3, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

        # Learnable tokens and embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches + 1, embed_dim)
        )
        self.temporal_embed = nn.Parameter(
            torch.zeros(1, num_frames, embed_dim)
        )

        # Embedding dropout
        self.embed_drop = nn.Dropout(dropout)

        # Transformer blocks (full 12 blocks)
        self.blocks = nn.ModuleList([
            TimeSformerBlock(embed_dim, num_heads, dropout=dropout)
            for _ in range(depth)
        ])

        # Final norm (required for Pre-LN transformers)
        self.norm = nn.LayerNorm(embed_dim)

        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.temporal_embed, std=0.02)

    def forward(self, x):
        b, t, c, h, w = x.shape

        # Patch embed each frame
        x = rearrange(x, "b t c h w -> (b t) c h w")
        x = self.patch_embed(x)          # (B*T, D, H', W')
        x = rearrange(x, "bt d h w -> bt (h w) d")

        # Prepend CLS token to each frame
        cls = self.cls_token.expand(b * t, -1, -1)
        x = torch.cat([cls, x], dim=1)   # (B*T, N+1, D)

        # Add spatial position embedding
        x = x + self.pos_embed

        # Reshape to (B, T, N+1, D)
        x = rearrange(x, "(b t) n d -> b t n d", b=b)

        # Add temporal embedding (broadcast over patches)
        x = x + self.temporal_embed.unsqueeze(2)

        # Embedding dropout
        x = self.embed_drop(x)

        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x)

        # Classification: average CLS tokens across all frames
        cls_tokens = x[:, :, 0, :]       # (B, T, D)
        cls_avg = cls_tokens.mean(dim=1)  # (B, D)

        # Final norm + head
        logits = self.head(self.norm(cls_avg))
        return logits

## 3. Pretrained ViT Weight Loading

Load pretrained ViT-Base/16 and transfer **all 12 blocks** to TimeSformer.

- Spatial attention, norms, MLP: from ViT (pretrained on ImageNet)
- Temporal attention: randomly initialized (new capability that ViT doesn't have)

In [ ]:
# Load pretrained ViT-Base/16
vit = timm.create_model('vit_base_patch16_224', pretrained=True)
vit.eval()
print("✓ Loaded pretrained ViT-Base/16")

In [ ]:
def load_vit_weights_into_timesformer(timesformer, vit):
    """Transfer pretrained ViT weights to TimeSformer.

    Maps ViT block i → TimeSformer block i (all 12 blocks).
    Temporal attention is left randomly initialized (new capability).
    """
    # ── Patch embedding ──
    timesformer.patch_embed.weight.data.copy_(
        vit.patch_embed.proj.weight.data
    )
    timesformer.patch_embed.bias.data.copy_(
        vit.patch_embed.proj.bias.data
    )

    # ── CLS token ──
    timesformer.cls_token.data.copy_(vit.cls_token.data)

    # ── Position embedding ──
    timesformer.pos_embed.data.copy_(vit.pos_embed.data)

    # ── Final LayerNorm ──
    timesformer.norm.weight.data.copy_(vit.norm.weight.data)
    timesformer.norm.bias.data.copy_(vit.norm.bias.data)

    # ── Transformer blocks ──
    loaded = 0
    for ts_block, vit_block in zip(timesformer.blocks, vit.blocks):
        # Spatial attention ← ViT attention
        ts_block.spatial_attn.in_proj_weight.data.copy_(
            vit_block.attn.qkv.weight.data
        )
        ts_block.spatial_attn.in_proj_bias.data.copy_(
            vit_block.attn.qkv.bias.data
        )
        ts_block.spatial_attn.out_proj.weight.data.copy_(
            vit_block.attn.proj.weight.data
        )
        ts_block.spatial_attn.out_proj.bias.data.copy_(
            vit_block.attn.proj.bias.data
        )

        # norm2 (spatial) ← ViT norm1 (attention norm)
        ts_block.norm2.weight.data.copy_(vit_block.norm1.weight.data)
        ts_block.norm2.bias.data.copy_(vit_block.norm1.bias.data)

        # norm3 (MLP) ← ViT norm2 (MLP norm)
        ts_block.norm3.weight.data.copy_(vit_block.norm2.weight.data)
        ts_block.norm3.bias.data.copy_(vit_block.norm2.bias.data)

        # MLP
        ts_block.mlp[0].weight.data.copy_(vit_block.mlp.fc1.weight.data)
        ts_block.mlp[0].bias.data.copy_(vit_block.mlp.fc1.bias.data)
        ts_block.mlp[3].weight.data.copy_(vit_block.mlp.fc2.weight.data)
        ts_block.mlp[3].bias.data.copy_(vit_block.mlp.fc2.bias.data)

        loaded += 1

    print(f"✓ Loaded ViT weights → TimeSformer (all {loaded} blocks)")
    print(f"  ✓ Patch embed, CLS token, position embed, final norm")
    print(f"  ✓ Spatial attention + LayerNorms + MLP ({loaded} blocks)")
    print(f"  ✗ Temporal attention — randomly initialized (new capability)")
    print(f"    (ViT only understands single images; temporal attention lets")
    print(f"     the model relate patches ACROSS frames, learning motion)")

## 4. Data Loading

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/TimeSformer/frames/'
NUM_FRAMES = 8
BATCH_SIZE = 4

train_dataset = VideoFrameDataset(
    DATA_ROOT, num_frames=NUM_FRAMES, split='train'
)
val_dataset = VideoFrameDataset(
    DATA_ROOT, num_frames=NUM_FRAMES, split='val'
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2
)

print(f"Classes: {train_dataset.classes}")
print(f"Train: {len(train_dataset)} videos")
print(f"Val:   {len(val_dataset)} videos")

# Show class distribution
from collections import Counter
train_labels = [s[1] for s in train_dataset.samples]
val_labels = [s[1] for s in val_dataset.samples]
print(f"\nTrain class distribution:")
for cls_idx, count in sorted(Counter(train_labels).items()):
    print(f"  {train_dataset.classes[cls_idx]:20s}: {count}")
print(f"\nVal class distribution:")
for cls_idx, count in sorted(Counter(val_labels).items()):
    print(f"  {val_dataset.classes[cls_idx]:20s}: {count}")

## 5. Model, Optimizer, Scheduler

In [ ]:
model = TimeSformer(
    num_classes=len(train_dataset.classes),
    num_frames=NUM_FRAMES,
    embed_dim=768,
    depth=12,       # Full depth — keeps all pretrained semantic layers
    num_heads=12,
    dropout=0.15    # Slightly higher dropout to compensate for full depth
).to(device)

# Load pretrained weights (all 12 ViT blocks)
load_vit_weights_into_timesformer(model, vit)

# Label smoothing helps prevent overconfident predictions
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")

## 6. Training + Validation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for videos, labels in loader:
        videos = videos.to(device)
        labels = labels.to(device)

        logits = model(videos)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model on a dataset."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for videos, labels in loader:
        videos = videos.to(device)
        labels = labels.to(device)

        logits = model(videos)
        loss = criterion(logits, labels)

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / len(loader), correct / total, all_preds, all_labels

## 7. Stage 1 — Freeze Pretrained, Train Temporal + Head

Only train the **temporal attention** (new capability) and **classification head**.
All pretrained spatial/MLP/norm weights are frozen.

This lets the temporal attention learn to connect frames before we
do any fine-tuning of the pretrained spatial layers.

In [ ]:
# ── Stage 1: Freeze pretrained layers ──

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze: temporal attention + temporal norms + head + temporal embed
for block in model.blocks:
    for param in block.temporal_attn.parameters():
        param.requires_grad = True
    for param in block.norm1.parameters():  # temporal norm
        param.requires_grad = True

for param in model.head.parameters():
    param.requires_grad = True
model.temporal_embed.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Stage 1 — Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
# ── Stage 1 Training ──

STAGE1_EPOCHS = 5
STAGE1_LR = 1e-4

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=STAGE1_LR, weight_decay=0.05
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=STAGE1_EPOCHS, eta_min=1e-6
)

best_val_acc = 0.0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("=" * 80)
print(f"  Stage 1: Temporal Attention + Head  |  {STAGE1_EPOCHS} epochs  |  LR: {STAGE1_LR}")
print("=" * 80)

for epoch in range(STAGE1_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc, _, _ = evaluate(
        model, val_loader, criterion, device
    )
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_timesformer.pth')
        marker = " ★"

    print(
        f"  Epoch {epoch+1:2d}/{STAGE1_EPOCHS} │"
        f" Train Loss: {train_loss:.4f} │"
        f" Train Acc: {train_acc:.4f} │"
        f" Val Loss: {val_loss:.4f} │"
        f" Val Acc: {val_acc:.4f}{marker}"
    )

print(f"\n  Stage 1 Best Val Acc: {best_val_acc:.4f}")

## 8. Stage 2 — Unfreeze All, Fine-Tune End-to-End

Unfreeze all layers with a **lower learning rate** (2e-5).
Higher weight decay (0.05) and dropout (0.15) prevent overfitting.
Early stopping with patience=7 to let the model converge.

In [ ]:
# ── Stage 2: Unfreeze everything ──

for param in model.parameters():
    param.requires_grad = True

STAGE2_EPOCHS = 20
STAGE2_LR = 2e-5   # lower LR for full fine-tuning
PATIENCE = 7        # longer patience to allow convergence

optimizer = torch.optim.AdamW(
    model.parameters(), lr=STAGE2_LR, weight_decay=0.05
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=STAGE2_EPOCHS, eta_min=1e-6
)

no_improve = 0

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Stage 2 — All {trainable:,} parameters trainable")
print("=" * 80)
print(f"  Stage 2: Full Fine-Tuning  |  {STAGE2_EPOCHS} epochs  |  LR: {STAGE2_LR}  |  Patience: {PATIENCE}")
print("=" * 80)

for epoch in range(STAGE2_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc, _, _ = evaluate(
        model, val_loader, criterion, device
    )
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_timesformer.pth')
        marker = " ★"
        no_improve = 0
    else:
        no_improve += 1

    total_epoch = STAGE1_EPOCHS + epoch + 1
    print(
        f"  Epoch {total_epoch:2d}/{STAGE1_EPOCHS + STAGE2_EPOCHS} │"
        f" Train Loss: {train_loss:.4f} │"
        f" Train Acc: {train_acc:.4f} │"
        f" Val Loss: {val_loss:.4f} │"
        f" Val Acc: {val_acc:.4f}{marker}"
    )

    # Early stopping
    if no_improve >= PATIENCE:
        print(f"\n  ⚠ Early stopping at epoch {total_epoch} (no improvement for {PATIENCE} epochs)")
        break

print("=" * 80)
print(f"  ✓ Best Validation Accuracy: {best_val_acc:.4f}")
print("=" * 80)

## 9. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

total_epochs = len(train_losses)
epochs_range = range(1, total_epochs + 1)

# Loss
ax1.plot(epochs_range, train_losses, label='Train Loss',
         color='#e07b54', linewidth=2, marker='o', markersize=4)
ax1.plot(epochs_range, val_losses, label='Val Loss',
         color='#5ba8e0', linewidth=2, marker='s', markersize=4)
ax1.axvline(x=STAGE1_EPOCHS + 0.5, color='gray', linestyle='--',
            alpha=0.5, label='Stage 1 → 2')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(epochs_range, train_accs, label='Train Acc',
         color='#e07b54', linewidth=2, marker='o', markersize=4)
ax2.plot(epochs_range, val_accs, label='Val Acc',
         color='#5ba8e0', linewidth=2, marker='s', markersize=4)
ax2.axvline(x=STAGE1_EPOCHS + 0.5, color='gray', linestyle='--',
            alpha=0.5, label='Stage 1 → 2')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Evaluation — Per-Class Accuracy + Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Load best model
model.load_state_dict(torch.load('best_timesformer.pth'))

# Evaluate
val_loss, val_acc, all_preds, all_labels = evaluate(
    model, val_loader, criterion, device
)

class_names = train_dataset.classes

print("=" * 60)
print("  Evaluation Results (Best Model)")
print("=" * 60)
print(f"  Validation Accuracy: {val_acc:.4f}")
print(f"  Validation Loss:     {val_loss:.4f}")
print()
print(classification_report(
    all_labels, all_preds, target_names=class_names
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax, square=True,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 14, 'weight': 'bold'}
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix — TimeSformer (V2)', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Sample Predictions Visualization

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, device, num_samples=6):
    """Visualize predictions: show 8 frames per video with true/predicted labels."""
    model.eval()

    # ImageNet denormalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    num_samples = min(num_samples, len(dataset))
    indices = torch.randperm(len(dataset))[:num_samples]

    fig, axes = plt.subplots(
        num_samples, 8,
        figsize=(20, 3 * num_samples)
    )

    if num_samples == 1:
        axes = axes.reshape(1, -1)

    for row, idx in enumerate(indices):
        video, label = dataset[idx.item()]
        logits = model(video.unsqueeze(0).to(device))
        pred = torch.argmax(logits, dim=1).item()

        true_name = dataset.classes[label]
        pred_name = dataset.classes[pred]
        is_correct = pred == label
        color = '#2ecc71' if is_correct else '#e74c3c'
        symbol = '✓' if is_correct else '✗'

        for col in range(8):
            frame = video[col] * std + mean  # Denormalize
            frame = frame.permute(1, 2, 0).clamp(0, 1).numpy()
            axes[row, col].imshow(frame)
            axes[row, col].axis('off')

            if col == 0:
                axes[row, col].set_title(
                    f"{symbol} True: {true_name}\nPred: {pred_name}",
                    fontsize=9, color=color, fontweight='bold'
                )

    plt.suptitle(
        'Sample Predictions (8 uniformly sampled frames per video)',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

show_predictions(model, val_dataset, device, num_samples=6)